Add a new feature called “Haunted Places Witness Count” and use Number
Parser :
https://github.com/scrapinghub/number
-
parser to obtain the number of witnesses (if possible to identify in the description). If unable to identify, set count to 0.

In [1]:
!pip install number-parser

In [1]:
import pandas as pd
import numpy as np
from number_parser import parse
from number_parser import parse_number
import re

In [2]:
haunted_places = pd.read_csv("../data/haunted_places.tsv", sep='\t')

In [3]:
def extract_witness_count(description):
    text = description.lower()
    match_count = 0
    num_text = parse(text)
    strict_matches = re.findall(r'(\d+\s*(?:people|witnesses|individuals|sisters|students|civilians|teachers|eyewitnesses|employees|workers|teens|teenagers))', num_text)
    if strict_matches:
        for match in strict_matches:
            number_text = match[0]
            match_count += int(number_text)
        return match_count
    witnesses = re.findall(r'(people|witnesses|individuals|sisters|students|civilians|teachers|eyewitnesses|employees|workers|teens|teenagers|crowd)', num_text)
    for witness in witnesses:
        match_count += 2
    witnessed = re.search(r'(seen|saw|heard|reported|witnessed)', num_text)
    if witnessed:
        if match_count > 0:
            return match_count
        else:
            return 1
    return 0
        

In [4]:
haunted_places['Haunted Places Witness Count'] = haunted_places['description'].apply(extract_witness_count)

In [7]:
# Count the number of entries with witness count = 0 and witness count > 0
zero_witness_count = (haunted_places['Haunted Places Witness Count'] == 0).sum()
non_zero_witness_count = (haunted_places['Haunted Places Witness Count'] > 0).sum()

# Calculate percentages
total = len(haunted_places)
zero_witness_percentage = (zero_witness_count / total) * 100
non_zero_witness_percentage = (non_zero_witness_count / total) * 100

# Print the results
print(f"Number of entries with witness count = 0: {zero_witness_count} ({zero_witness_percentage:.2f}%)")
print(f"Number of entries with witness count > 0: {non_zero_witness_count} ({non_zero_witness_percentage:.2f}%)")

Number of entries with witness count = 0: 6005 (54.63%)
Number of entries with witness count > 0: 4987 (45.37%)


In [8]:
haunted_places.to_csv("../data/haunted_places.tsv", sep='\t', index=False)